In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split ,GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    matthews_corrcoef,
    brier_score_loss,
)

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 10
N_REPEATS = 5
DATA_PATH = "online_shoppers_intention.csv"
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    NOMINAL_INT_COLS = ["OperatingSystems", "Browser", "Region", "TrafficType"]
    df[NOMINAL_INT_COLS] = df[NOMINAL_INT_COLS].astype("category")
    df_encoded = pd.get_dummies(df, drop_first=True, dtype=int)
    X = df_encoded.drop(columns=["Revenue"])
    y = df_encoded["Revenue"]
    return X, y


In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """Standard binned ECE: weighted average gap between confidence and accuracy."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob > lo) & (y_prob <= hi) if lo > 0 else (y_prob >= lo) & (y_prob <= hi)
        if mask.sum() == 0:
            continue
        bin_conf = y_prob[mask].mean()
        bin_acc = y_true[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(bin_conf - bin_acc)
    return ece

In [ ]:
def best_threshold_by_f1(y_true, y_prob):
    """Select the probability threshold that maximizes F1 on the given (training-side) data."""
    thresholds = np.linspace(0.05, 0.95, 19)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t

In [ ]:


def tune_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)

    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    # Logistic Regression
    lr_grid = {"C": [0.01, 0.1, 1, 10, 100]}
    lr_search = GridSearchCV(
        LogisticRegression(class_weight="balanced", max_iter=20000, random_state=RANDOM_STATE),
        lr_grid, scoring="roc_auc", cv=inner_cv, n_jobs=-1
    )
    lr_search.fit(X_train_s, y_train)

    # Random Forest
    rf_grid = {
        "n_estimators": [200, 300, 500],
        "max_depth": [None, 10, 20, 30],
        "min_samples_leaf": [1, 3, 5],
        "max_features": ["sqrt", "log2"],
    }
    rf_search = RandomizedSearchCV(
        RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        rf_grid, n_iter=20, scoring="roc_auc", cv=inner_cv,
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_search.fit(X_train_s, y_train)

    print("Best LR params:", lr_search.best_params_)
    print("Best RF params:", rf_search.best_params_)
    return lr_search.best_params_, rf_search.best_params_

X, y = load_data()
BEST_LR_PARAMS, BEST_RF_PARAMS = tune_models(X, y)

In [ ]:
def fit_and_score(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_prob_test = model.predict_proba(X_test)[:, 1]

    # Default 0.5 threshold
    y_pred_default = (y_prob_test >= 0.5).astype(int)

    # Threshold tuned on an inner split of the TRAINING fold only (no test leakage)
    X_tr_inner, X_val_inner, y_tr_inner, y_val_inner = train_test_split(
        X_train, y_train, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train
    )
    inner_model = type(model)(**model.get_params())
    inner_model.fit(X_tr_inner, y_tr_inner)
    y_prob_val_inner = inner_model.predict_proba(X_val_inner)[:, 1]
    tuned_t = best_threshold_by_f1(y_val_inner, y_prob_val_inner)
    y_pred_tuned = (y_prob_test >= tuned_t).astype(int)

    return {
        "ROC_AUC": roc_auc_score(y_test, y_prob_test),
        "PR_AUC": average_precision_score(y_test, y_prob_test),
        "F1_default": f1_score(y_test, y_pred_default, zero_division=0),
        "MCC_default": matthews_corrcoef(y_test, y_pred_default),
        "F1_tuned": f1_score(y_test, y_pred_tuned, zero_division=0),
        "MCC_tuned": matthews_corrcoef(y_test, y_pred_tuned),
        "Tuned_threshold": tuned_t,
        "Brier": brier_score_loss(y_test, y_prob_test),
        "ECE": expected_calibration_error(y_test.values, y_prob_test),
    }

In [ ]:

def main(best_lr_params, best_rf_params):
    X, y = load_data()

    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

    fold_rows = []
    for fold_idx, (train_idx, test_idx) in enumerate(rskf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        # --- Logistic Regression: baseline ---
        lr_base = LogisticRegression(class_weight="balanced", max_iter=20000, random_state=RANDOM_STATE)
        s = fit_and_score(lr_base, X_train_s, y_train, X_test_s, y_test)
        s.update({"Fold": fold_idx, "Model": "Logistic Regression (baseline)"})
        fold_rows.append(s)

        # --- Logistic Regression: tuned ---
        lr_tuned = LogisticRegression(class_weight="balanced", max_iter=20000,
                                       random_state=RANDOM_STATE, **best_lr_params)
        s = fit_and_score(lr_tuned, X_train_s, y_train, X_test_s, y_test)
        s.update({"Fold": fold_idx, "Model": "Logistic Regression (tuned)"})
        fold_rows.append(s)

        # --- Random Forest: baseline ---
        rf_base = RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1
        )
        s = fit_and_score(rf_base, X_train_s, y_train, X_test_s, y_test)
        s.update({"Fold": fold_idx, "Model": "Random Forest (baseline)"})
        fold_rows.append(s)

        # --- Random Forest: tuned ---
        rf_tuned = RandomForestClassifier(
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, **best_rf_params
        )
        s = fit_and_score(rf_tuned, X_train_s, y_train, X_test_s, y_test)
        s.update({"Fold": fold_idx, "Model": "Random Forest (tuned)"})
        fold_rows.append(s)

        if (fold_idx + 1) % 10 == 0:
            print(f"Completed fold {fold_idx + 1}/{N_SPLITS * N_REPEATS}")

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(os.path.join(OUT_DIR, "repeated_cv_fold_scores.csv"), index=False)

    metrics = ["ROC_AUC", "PR_AUC", "F1_default", "MCC_default", "F1_tuned", "MCC_tuned", "Brier", "ECE"]
    model_names = [
        "Logistic Regression (baseline)", "Logistic Regression (tuned)",
        "Random Forest (baseline)", "Random Forest (tuned)",
    ]

    # ---- Summary: mean, std, normal-approx 95% CI, and bootstrap 95% CI ----
    summary_rows = []
    rng = np.random.default_rng(RANDOM_STATE)
    for model_name in model_names:
        sub = fold_df[fold_df.Model == model_name]
        for metric in metrics:
            vals = sub[metric].values
            mean = vals.mean()
            std = vals.std(ddof=1)
            se = std / np.sqrt(len(vals))
            ci_lo_normal = mean - 1.96 * se
            ci_hi_normal = mean + 1.96 * se

            boot_means = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(2000)]
            ci_lo_boot, ci_hi_boot = np.percentile(boot_means, [2.5, 97.5])

            summary_rows.append({
                "Model": model_name,
                "Metric": metric,
                "Mean": round(mean, 4),
                "Std": round(std, 4),
                "N_folds": len(vals),
                "CI95_normal_low": round(ci_lo_normal, 4),
                "CI95_normal_high": round(ci_hi_normal, 4),
                "CI95_bootstrap_low": round(ci_lo_boot, 4),
                "CI95_bootstrap_high": round(ci_hi_boot, 4),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(OUT_DIR, "repeated_cv_summary.csv"), index=False)
    print("\n=== Repeated stratified CV summary (mean, std, 95% CI) ===")
    print(summary_df.to_string(index=False))

    # ---- Paired tests, fold-wise ----
    def paired_test_block(name_a, name_b, label):
        sub_a = fold_df[fold_df.Model == name_a].sort_values("Fold").reset_index(drop=True)
        sub_b = fold_df[fold_df.Model == name_b].sort_values("Fold").reset_index(drop=True)
        rows = []
        for metric in metrics:
            diffs = sub_a[metric].values - sub_b[metric].values
            try:
                wilcoxon_stat, wilcoxon_p = stats.wilcoxon(diffs)
            except ValueError:
                wilcoxon_stat, wilcoxon_p = np.nan, np.nan
            ttest_stat, ttest_p = stats.ttest_rel(sub_a[metric].values, sub_b[metric].values)
            n = len(diffs)
            effect_size_r = ttest_stat / np.sqrt(n) if n > 0 else np.nan

            rows.append({
                "Comparison": label,
                "Metric": metric,
                f"Mean_diff({name_a}_minus_{name_b})": round(diffs.mean(), 4),
                "Std_diff": round(diffs.std(ddof=1), 4),
                "Wilcoxon_stat": round(wilcoxon_stat, 4) if not np.isnan(wilcoxon_stat) else np.nan,
                "Wilcoxon_p": round(wilcoxon_p, 6) if not np.isnan(wilcoxon_p) else np.nan,
                "Paired_ttest_stat": round(ttest_stat, 4),
                "Paired_ttest_p": round(ttest_p, 6),
                "Significant_at_0.05_Wilcoxon": (wilcoxon_p < 0.05) if not np.isnan(wilcoxon_p) else None,
            })
        return rows

    paired_rows = []
    # Primary comparison for the paper: tuned RF vs tuned LR
    paired_rows += paired_test_block(
        "Random Forest (tuned)", "Logistic Regression (tuned)", "RF(tuned) vs LR(tuned)"
    )
    # Does tuning actually help each model? (answers reviewer's "fair comparison" concern directly)
    paired_rows += paired_test_block(
        "Logistic Regression (tuned)", "Logistic Regression (baseline)", "LR: tuned vs baseline"
    )
    paired_rows += paired_test_block(
        "Random Forest (tuned)", "Random Forest (baseline)", "RF: tuned vs baseline"
    )

    paired_df = pd.DataFrame(paired_rows)
    paired_df.to_csv(os.path.join(OUT_DIR, "paired_test_results.csv"), index=False)
    print("\n=== Paired tests (fold-wise, RepeatedStratifiedKFold) ===")
    print(paired_df.to_string(index=False))

    print(f"\nSaved: {OUT_DIR}/repeated_cv_fold_scores.csv, {OUT_DIR}/repeated_cv_summary.csv, "
          f"{OUT_DIR}/paired_test_results.csv")
    
    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

    fold_rows = []
    for fold_idx, (train_idx, test_idx) in enumerate(rskf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        # --- Logistic Regression: baseline ---
        lr_base = LogisticRegression(class_weight="balanced", max_iter=20000, random_state=RANDOM_STATE)
        s = fit_and_score(lr_base, X_train_s, y_train, X_test_s, y_test)
        s.update({"Fold": fold_idx, "Model": "Logistic Regression (baseline)"})
        fold_rows.append(s)

        # --- Logistic Regression: tuned ---
        lr_tuned = LogisticRegression(class_weight="balanced", max_iter=20000,
                                       random_state=RANDOM_STATE, **best_lr_params)
        s = fit_and_score(lr_tuned, X_train_s, y_train, X_test_s, y_test)
        s.update({"Fold": fold_idx, "Model": "Logistic Regression (tuned)"})
        fold_rows.append(s)

        # --- Random Forest: baseline ---
        rf_base = RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1
        )
        s = fit_and_score(rf_base, X_train_s, y_train, X_test_s, y_test)
        s.update({"Fold": fold_idx, "Model": "Random Forest (baseline)"})
        fold_rows.append(s)

        # --- Random Forest: tuned ---
        rf_tuned = RandomForestClassifier(
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, **best_rf_params
        )
        s = fit_and_score(rf_tuned, X_train_s, y_train, X_test_s, y_test)
        s.update({"Fold": fold_idx, "Model": "Random Forest (tuned)"})
        fold_rows.append(s)

        if (fold_idx + 1) % 10 == 0:
            print(f"Completed fold {fold_idx + 1}/{N_SPLITS * N_REPEATS}")

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(os.path.join(OUT_DIR, "repeated_cv_fold_scores.csv"), index=False)

    metrics = ["ROC_AUC", "PR_AUC", "F1_default", "MCC_default", "F1_tuned", "MCC_tuned", "Brier", "ECE"]
    model_names = [
        "Logistic Regression (baseline)", "Logistic Regression (tuned)",
        "Random Forest (baseline)", "Random Forest (tuned)",
    ]

    # ---- Summary: mean, std, normal-approx 95% CI, and bootstrap 95% CI ----
    summary_rows = []
    rng = np.random.default_rng(RANDOM_STATE)
    for model_name in model_names:
        sub = fold_df[fold_df.Model == model_name]
        for metric in metrics:
            vals = sub[metric].values
            mean = vals.mean()
            std = vals.std(ddof=1)
            se = std / np.sqrt(len(vals))
            ci_lo_normal = mean - 1.96 * se
            ci_hi_normal = mean + 1.96 * se

            boot_means = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(2000)]
            ci_lo_boot, ci_hi_boot = np.percentile(boot_means, [2.5, 97.5])

            summary_rows.append({
                "Model": model_name,
                "Metric": metric,
                "Mean": round(mean, 4),
                "Std": round(std, 4),
                "N_folds": len(vals),
                "CI95_normal_low": round(ci_lo_normal, 4),
                "CI95_normal_high": round(ci_hi_normal, 4),
                "CI95_bootstrap_low": round(ci_lo_boot, 4),
                "CI95_bootstrap_high": round(ci_hi_boot, 4),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(OUT_DIR, "repeated_cv_summary.csv"), index=False)
    print("\n=== Repeated stratified CV summary (mean, std, 95% CI) ===")
    print(summary_df.to_string(index=False))

    # ---- Paired tests, fold-wise ----
    def paired_test_block(name_a, name_b, label):
        sub_a = fold_df[fold_df.Model == name_a].sort_values("Fold").reset_index(drop=True)
        sub_b = fold_df[fold_df.Model == name_b].sort_values("Fold").reset_index(drop=True)
        rows = []
        for metric in metrics:
            diffs = sub_a[metric].values - sub_b[metric].values
            try:
                wilcoxon_stat, wilcoxon_p = stats.wilcoxon(diffs)
            except ValueError:
                wilcoxon_stat, wilcoxon_p = np.nan, np.nan
            ttest_stat, ttest_p = stats.ttest_rel(sub_a[metric].values, sub_b[metric].values)
            n = len(diffs)
            effect_size_r = ttest_stat / np.sqrt(n) if n > 0 else np.nan

            rows.append({
                "Comparison": label,
                "Metric": metric,
                f"Mean_diff({name_a}_minus_{name_b})": round(diffs.mean(), 4),
                "Std_diff": round(diffs.std(ddof=1), 4),
                "Wilcoxon_stat": round(wilcoxon_stat, 4) if not np.isnan(wilcoxon_stat) else np.nan,
                "Wilcoxon_p": round(wilcoxon_p, 6) if not np.isnan(wilcoxon_p) else np.nan,
                "Paired_ttest_stat": round(ttest_stat, 4),
                "Paired_ttest_p": round(ttest_p, 6),
                "Significant_at_0.05_Wilcoxon": (wilcoxon_p < 0.05) if not np.isnan(wilcoxon_p) else None,
            })
        return rows

    paired_rows = []
    # Primary comparison for the paper: tuned RF vs tuned LR
    paired_rows += paired_test_block(
        "Random Forest (tuned)", "Logistic Regression (tuned)", "RF(tuned) vs LR(tuned)"
    )
    # Does tuning actually help each model? (answers reviewer's "fair comparison" concern directly)
    paired_rows += paired_test_block(
        "Logistic Regression (tuned)", "Logistic Regression (baseline)", "LR: tuned vs baseline"
    )
    paired_rows += paired_test_block(
        "Random Forest (tuned)", "Random Forest (baseline)", "RF: tuned vs baseline"
    )

    paired_df = pd.DataFrame(paired_rows)
    paired_df.to_csv(os.path.join(OUT_DIR, "paired_test_results.csv"), index=False)
    print("\n=== Paired tests (fold-wise, RepeatedStratifiedKFold) ===")
    print(paired_df.to_string(index=False))

    print(f"\nSaved: {OUT_DIR}/repeated_cv_fold_scores.csv, {OUT_DIR}/repeated_cv_summary.csv, "
          f"{OUT_DIR}/paired_test_results.csv")



In [ ]:
if __name__ == "__main__":
    main(BEST_LR_PARAMS, BEST_RF_PARAMS)
